# Problem Set 4: SIR Modeling

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due 10/21 (beginning of Wednesday lecture)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Simulating epidemic dynamics with the SIR model.

**Before you do anything else:** go to **File → Save a copy in Drive**.

> 📝 **DRAFT for instructor review** — questions and emphasis to be finalized with Prof. Schwandt.

## Part 0: Set-up

In this problem set you will build and experiment with the **SIR model** — the workhorse model of epidemiology discussed in class. There is no dataset this week: you will *generate* the epidemics yourself and study how they respond to the model's parameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

## Part 1: The model

The SIR model divides a population of size $N$ into three groups that change over time:

- $S$ — **Susceptible**: can catch the disease
- $I$ — **Infected**: currently infectious
- $R$ — **Recovered**: immune (or deceased) — no longer susceptible or infectious

Each day, two things happen:

1. **New infections:** each infected person meets others at random; the chance a meeting is with a susceptible person is $S/N$. With transmission rate $\beta$ (beta), the number of new infections per day is $\beta \cdot I \cdot S / N$.
2. **Recoveries:** a fraction $\gamma$ (gamma) of infected people recover each day (so the average infectious period lasts $1/\gamma$ days).

The famous **basic reproduction number** is $R_0 = \beta / \gamma$: the number of people one infected person infects, on average, in a fully susceptible population.

The function below simulates this day by day. Read it line by line — it is the whole model.

In [ ]:
def sir(beta, gamma, N=10_000, I0=10, days=200):
    """Simulate an SIR epidemic day by day. Returns a DataFrame with columns S, I, R."""
    S, I, R = N - I0, I0, 0
    rows = []
    for day in range(days + 1):
        rows.append({"day": day, "S": S, "I": I, "R": R})
        new_infections = beta * I * S / N
        new_recoveries = gamma * I
        S = S - new_infections
        I = I + new_infections - new_recoveries
        R = R + new_recoveries
    return pd.DataFrame(rows)

# Baseline epidemic: beta = 0.3, gamma = 0.1  =>  R0 = 3
baseline = sir(beta=0.3, gamma=0.1)

plt.figure(figsize=(8, 5))
for col, label in [("S", "Susceptible"), ("I", "Infected"), ("R", "Recovered")]:
    plt.plot(baseline.day, baseline[col], label=label)
plt.xlabel("Day"); plt.ylabel("People"); plt.legend()
plt.title("Baseline epidemic: $\\beta=0.3$, $\\gamma=0.1$ ($R_0=3$)")
plt.show()

print(f"Peak infected: {baseline.I.max():,.0f} on day {baseline.I.idxmax()}")
print(f"Never infected (S at end): {baseline.S.iloc[-1]:,.0f} of 10,000")

**Question 1.** Describe the three curves in words. The epidemic clearly ends — but notice that some people are *never* infected. Why does the epidemic die out **before** everyone has been infected? (Hint: what has to be true about $S/N$ for infections to keep growing?)

**Answer:** *[your answer here]*

## Part 2: How $R_0$ shapes the epidemic

Keep $\gamma = 0.1$ and vary $\beta$, which varies $R_0$.

In [ ]:
plt.figure(figsize=(8, 5))
results = []
for beta in [0.12, 0.2, 0.3, 0.5]:
    epi = sir(beta=beta, gamma=0.1, days=300)
    plt.plot(epi.day, epi.I, label=f"$R_0$ = {beta/0.1:.1f}")
    results.append({"R0": beta / 0.1,
                    "peak infected": round(epi.I.max()),
                    "peak day": int(epi.I.idxmax()),
                    "total ever infected": round(epi.R.iloc[-1] + epi.I.iloc[-1])})
plt.xlabel("Day"); plt.ylabel("Infected"); plt.legend()
plt.title("Infected over time for different $R_0$")
plt.show()

pd.DataFrame(results)

**Question 2a.** How do the **peak** of the epidemic and the **total number ever infected** change with $R_0$? What happens when $R_0$ is close to 1?

**Answer:** *[your answer here]*

**Question 2b.** The **herd immunity threshold** is the share of the population that must be immune for the epidemic to stop growing: $1 - 1/R_0$. Compute it (by hand or in a cell) for $R_0 = 1.2$, $R_0 = 3$, and $R_0 = 15$ (roughly measles). What does this imply about how contagious a disease can be before vaccination campaigns become very demanding?

**Answer:** *[your answer here]*

## Part 3: "Flattening the curve"

Suppose that on **day 40** the government introduces distancing and masking, which cuts the transmission rate in half (from 0.3 to 0.15) from that day on. The function below allows $\beta$ to change mid-epidemic.

In [ ]:
def sir_intervention(beta_before, beta_after, switch_day, gamma, N=10_000, I0=10, days=300):
    S, I, R = N - I0, I0, 0
    rows = []
    for day in range(days + 1):
        rows.append({"day": day, "S": S, "I": I, "R": R})
        beta = beta_before if day < switch_day else beta_after
        new_infections = beta * I * S / N
        new_recoveries = gamma * I
        S, I, R = S - new_infections, I + new_infections - new_recoveries, R + new_recoveries
    return pd.DataFrame(rows)

no_policy   = sir(beta=0.3, gamma=0.1, days=300)
with_policy = sir_intervention(0.3, 0.15, switch_day=40, gamma=0.1)

plt.figure(figsize=(8, 5))
plt.plot(no_policy.day, no_policy.I, label="No intervention")
plt.plot(with_policy.day, with_policy.I, label="Distancing from day 40")
plt.axhline(1000, color="gray", linestyle="--", label="Hospital capacity (say)")
plt.xlabel("Day"); plt.ylabel("Infected"); plt.legend()
plt.show()

for name, epi in [("No intervention", no_policy), ("With intervention", with_policy)]:
    print(f"{name:20s} peak {epi.I.max():7,.0f} on day {epi.I.idxmax():3d}; "
          f"ever infected {epi.R.iloc[-1] + epi.I.iloc[-1]:,.0f}")

**Question 3.** Describe what the intervention does to (i) the height and timing of the peak and (ii) the total number ever infected. Using the dashed "hospital capacity" line, explain in your own words the public-health logic of "flattening the curve" — even for an intervention that does not, by itself, end the epidemic.

**Answer:** *[your answer here]*

## Part 4: Vaccination

Vaccinating a fraction $v$ of the population *before* the epidemic moves those people straight into $R$ at day 0. Experiment with different values of $v$ (for $R_0 = 3$): try 0.3, 0.5, 0.667, 0.8.

In [ ]:
def sir_vaccinated(v, beta=0.3, gamma=0.1, N=10_000, I0=10, days=300):
    S, I, R = N * (1 - v) - I0, I0, N * v
    rows = []
    for day in range(days + 1):
        rows.append({"day": day, "S": S, "I": I, "R": R})
        new_infections = beta * I * S / N
        new_recoveries = gamma * I
        S, I, R = S - new_infections, I + new_infections - new_recoveries, R + new_recoveries
    return pd.DataFrame(rows)

plt.figure(figsize=(8, 5))
for v in [0.0, 0.3, 0.5, 0.667, 0.8]:
    epi = sir_vaccinated(v)
    plt.plot(epi.day, epi.I, label=f"{v:.0%} vaccinated")
plt.xlabel("Day"); plt.ylabel("Infected"); plt.legend()
plt.show()

**Question 4.** At which vaccination level does the epidemic essentially fail to take off? How does this relate to the herd immunity threshold you computed in Question 2b?

**Answer:** *[your answer here]*

**Question 5.** The SIR model makes strong simplifying assumptions — for instance, that everyone mixes with everyone at random and that $\beta$ is a fixed constant (unless we change it by hand). Name **two** ways these assumptions are violated in a real pandemic, and for each, explain how it could make real-world epidemic forecasts (like those debated during COVID-19) unreliable.

**Answer:** *[your answer here]*

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS04"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")